# 01 · 결정 모델의 확률과 평가
**대상:** 딥러닝 경험자 · **시간:** 15분 · **환경:** Python 3.11+ CPU, 표준 라이브러리만 사용.

목표: `noul` / `choice` / `score` 해석, accuracy와 calibration 구분, validation에서 보류 임계값 선택.
이 노트북의 모든 로짓·확률은 **교육용 합성 값**이다. Jev 또는 학습 모델의 추론 결과가 아니다.

순서: 1. 확률 변환 → 2. 질문 유형 → 3. 평가 → 4. 온도 보정 → 5. 보류 정책.


In [ ]:
import math

def softmax(logits, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    z = [v / temperature for v in logits]
    exps = [math.exp(v - max(z)) for v in z]
    return [v / sum(exps) for v in exps]

def metrics(probabilities, labels, bins=5):
    if not probabilities or len(probabilities) != len(labels):
        raise ValueError("non-empty aligned predictions and labels required")
    if bins < 1:
        raise ValueError("bins must be positive")
    for p, y in zip(probabilities, labels):
        if (not p or not all(math.isfinite(v) and 0 <= v <= 1 for v in p)
                or not math.isclose(sum(p), 1, abs_tol=1e-6)
                or not isinstance(y, int) or not 0 <= y < len(p)):
            raise ValueError("invalid distribution or label")
    predicted = [max(range(len(p)), key=p.__getitem__) for p in probabilities]
    correct = [int(a == b) for a, b in zip(predicted, labels)]
    confidence = [max(p) for p in probabilities]
    n = len(labels)
    nll = -sum(math.log(max(p[y], 1e-12)) for p, y in zip(probabilities, labels)) / n
    # Multiclass Brier: sum over classes, then mean over samples (NOT divided by K).
    brier = sum(sum((v - int(k == y)) ** 2 for k, v in enumerate(p))
                for p, y in zip(probabilities, labels)) / n
    reliability = []
    for b in range(bins):
        ids = [i for i, c in enumerate(confidence) if min(int(c * bins), bins - 1) == b]
        if ids:
            acc = sum(correct[i] for i in ids) / len(ids)
            conf = sum(confidence[i] for i in ids) / len(ids)
            reliability.append((b, len(ids), acc, conf))
    ece = sum(count / n * abs(acc - conf) for _, count, acc, conf in reliability)
    return dict(accuracy=sum(correct)/n, nll=nll, brier=brier, ece=ece,
                reliability=reliability)


## 1. 선택지 로짓을 확률로
$p_k=\exp(z_k/T)/\sum_j\exp(z_j/T)$. 양의 스칼라 온도는 argmax를 바꾸지 않지만 확률의 크기를 바꾼다.
훈련에서 soft label $q$를 쓰면 CE는 $-\sum_k q_k\log p_k$다.


In [ ]:
labels = ['billing', 'technical', 'sales']
logits = [2.2, 0.8, -0.3]
for temperature in [0.5, 1.0, 2.0]:
    print(temperature, dict(zip(labels, [round(p, 4) for p in softmax(logits, temperature)])))
q = [0.7, 0.2, 0.1]
p = softmax(logits)
print('soft-target CE:', -sum(a * math.log(b) for a, b in zip(q, p)))


## 2. 세 질문 유형
`noul=0.8`은 yes 확률이다. `score`는 순서 있는 등급의 **기댓값**이며 argmax 등급과 다르다.
서비스의 `confidence` 필드는 모델별 정의가 다르므로 선택지 확률과 혼동하지 않는다.


In [ ]:
p_yes = 0.8
urgency = {'0': 0.1, '1': 0.2, '2': 0.7}
expected_score = sum(int(k) * v for k, v in urgency.items())
print('noul:', p_yes, '| no:', 1 - p_yes)
print('score expectation:', expected_score, '| most likely level:', max(urgency, key=urgency.get))
assert math.isclose(expected_score, 1.6)


## 3. Accuracy만 같으면 같은 모델일까?
아래 두 예측은 모두 75% 정확도지만 잘못된 예측의 확신이 다르다.
여기서는 hard label NLL, 클래스 합 방식 Brier, top-label ECE(5개 고정 폭 bin)를 사용한다.
공식 데이터의 soft-gold 지표와 수치를 직접 비교하지 않는다. 작은 표본의 ECE는 특히 불안정하다.


In [ ]:
y = [0, 1, 0, 1]
moderate = [[.7, .3], [.3, .7], [.7, .3], [.7, .3]]
overconfident = [[.99, .01], [.01, .99], [.99, .01], [.99, .01]]
for name, probs in [('moderate', moderate), ('overconfident', overconfident)]:
    print(name, metrics(probs, y))
assert metrics(overconfident, y)['nll'] > metrics(moderate, y)['nll']


## 4. 보정 데이터에서만 온도 선택
별도 calibration 데이터의 NLL로 T를 선택하고, 고정한 T를 test에 적용한다.
합성 예시에서 개선되더라도 실제 데이터에서의 개선을 보장하지 않는다.
Unsloth의 실제 보정은 02에서 `FastDecisionModel.calibrate`로 수행한다.


In [ ]:
cal_logits = [[4., 0.], [0., 4.], [4., 0.], [4., 0.]]
cal_labels = [0, 1, 0, 1]
test_logits = [[3., 0.], [0., 3.], [3., 0.], [0., 3.]]
test_labels = [0, 1, 1, 1]
candidates = [0.5, 1., 1.5, 2., 3., 4., 6.]
best_t = min(candidates, key=lambda t: metrics([softmax(z, t) for z in cal_logits], cal_labels)['nll'])
print('calibration-selected T:', best_t)
for t in [1., best_t]:
    print('test, T=', t, metrics([softmax(z, t) for z in test_logits], test_labels))


## 5. 실습: coverage와 선택적 정확도
확률이 임계값보다 작은 입력은 사람 검토로 보낸다고 가정한다. 다음 함수에서 coverage는 자동 처리 비율,
selective accuracy는 자동 처리된 사례에서의 정답률이다. 한 건도 없으면 정확도는 `None`이다.

**문제:** validation에서 정확도 90% 이상을 만족하면서 coverage가 가장 큰 임계값을 고르자.
이 보장은 작은 validation 표본 안에서만 성립한다. 실제 서비스에는 독립 테스트와 충분한 표본이 필요하다.


In [ ]:
def selective_metrics(probs, labels, threshold):
    ids = [i for i, p in enumerate(probs) if max(p) >= threshold]
    correct = sum(max(range(len(probs[i])), key=probs[i].__getitem__) == labels[i] for i in ids)
    return {'coverage': len(ids)/len(probs), 'accuracy': correct/len(ids) if ids else None}

validation_probs = [[.95,.05], [.1,.9], [.8,.2], [.4,.6], [.55,.45]]
validation_labels = [0,1,1,1,0]
results = [(t, selective_metrics(validation_probs, validation_labels, t)) for t in [.5,.7,.85,.95,1.]]
print(results)
# 해설: 통과 후보 중 coverage 최대. 테스트 라벨을 이 선택에 사용하지 않는다.
feasible = [(t, r) for t, r in results if r['accuracy'] is not None and r['accuracy'] >= .9]
chosen = max(feasible, key=lambda item: item[1]['coverage']) if feasible else None
print('selected:', chosen)
assert chosen[0] == .85
assert selective_metrics(validation_probs, validation_labels, 1.)['accuracy'] is None


## 다음 단계
02에서 합성 로짓 대신 실제 Qwen 결정 모델을 학습한다. 분포 이동, 한국어 데이터, 선택지 순서를 바꾸어
accuracy와 calibration이 함께 유지되는지 확인하자.

출처: [Unsloth 결정 모델](https://unsloth.ai/docs/new/train-your-own-decision-model-with-unsloth),
[typed-decisions 데이터 카드](https://huggingface.co/datasets/LocalLLaMA/typed-decisions).
